# Step 5a — MuseTalk real-time benchmark on a Kaggle T4

Question this answers: **can a free T4 lip-sync a face in real time?** MuseTalk 1.5 is run on its
own sample avatar and audio; we measure frames/second and GPU memory. Real-time needs **≥ 25 fps**
of generated frames (the avatar video's frame rate), with headroom.

Nothing here touches the agent worker. Settings: **GPU T4 x2**, **Internet On**. No secrets needed.

**Start from a fresh session** (Run → Factory reset, or stop and restart) if an earlier version of
this notebook ran: its OpenMMLab cell downgraded `requests`/`tqdm`/`urllib3`/`setuptools`.

MuseTalk's avatar preparation normally needs OpenMMLab (`mmpose`), which can't be installed on
Python 3.13. Instead, this notebook drops in `kaggle/musetalk_patches/preprocessing.py` from our
repo: same face box (MuseTalk's own S3FD) and the same 68-point landmarks, from `face-alignment`
(pure PyTorch). It runs once per avatar, not per frame, so it doesn't affect the speed measured.

In [ ]:
import sys, subprocess
print(sys.version)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch; print("torch", torch.__version__, "cuda", torch.version.cuda, torch.cuda.is_available())

In [ ]:
MT = "/kaggle/working/MuseTalk"
!rm -rf {MT} && git clone -q --depth 1 https://github.com/TMElyralab/MuseTalk.git {MT}
!cd {MT} && git log --oneline -1 && ls

In [ ]:
# MuseTalk's requirements.txt pins versions that don't exist for Python 3.13 (numpy 1.23,
# tensorflow 2.12); install the libraries it actually imports, unpinned, on top of Kaggle's torch.
!pip install -q diffusers accelerate transformers einops omegaconf librosa soundfile \
    opencv-python-headless "imageio[ffmpeg]" ffmpeg-python huggingface_hub gdown

In [ ]:
# Replace MuseTalk's mmpose-based landmark step with our face-alignment version.
!pip install -q face-alignment
import urllib.request
PATCH = ("https://raw.githubusercontent.com/daneshrooman-del/Rooman-Agent/"
         "feature/realtime-voice-loop/kaggle/musetalk_patches/preprocessing.py")
urllib.request.urlretrieve(PATCH, f"{MT}/musetalk/utils/preprocessing.py")
!grep -n "mmpose\|face_alignment" {MT}/musetalk/utils/preprocessing.py | head -5

In [ ]:
# Weights, laid out the way MuseTalk's download_weights.sh expects (models/...).
import os, urllib.request
from huggingface_hub import hf_hub_download
M = f"{MT}/models"
def hf(repo, filename, dest_dir, **kw):
    os.makedirs(dest_dir, exist_ok=True)
    return hf_hub_download(repo, filename, local_dir=dest_dir, **kw)

for f in ["musetalkV15/unet.pth", "musetalkV15/musetalk.json"]:
    hf("TMElyralab/MuseTalk", f, M)
for f in ["config.json", "diffusion_pytorch_model.bin"]:
    hf("stabilityai/sd-vae-ft-mse", f, f"{M}/sd-vae")
for f in ["config.json", "pytorch_model.bin", "preprocessor_config.json"]:
    hf("openai/whisper-tiny", f, f"{M}/whisper")
os.makedirs(f"{M}/face-parse-bisent", exist_ok=True)
!gdown -q --id 154JgKpzCPW82qINcVieuPH3fZ2e0P812 -O {M}/face-parse-bisent/79999_iter.pth
urllib.request.urlretrieve("https://download.pytorch.org/models/resnet18-5c106cde.pth",
                           f"{M}/face-parse-bisent/resnet18-5c106cde.pth")
!find {M} -type f | sed "s|{M}/||" | sort

In [ ]:
# Benchmark + profile: MuseTalk's own real-time script on its sample avatar and sample audio, with a
# timer on each of its two concurrent stages (our make_profile_script.py adds them to a copy of
# the script; the generated frames are identical). --skip_save_images = pure generation speed.
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/daneshrooman-del/Rooman-Agent/feature/realtime-voice-loop/"
    "kaggle/musetalk_patches/make_profile_script.py", f"{MT}/make_profile_script.py")
!cd {MT} && python make_profile_script.py
!cat {MT}/configs/inference/realtime.yaml

mem_log = "/kaggle/working/gpu_mem.csv"
sampler = subprocess.Popen(f"nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits -l 1 -i 0 > {mem_log}", shell=True)
# TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD: PyTorch >= 2.6 defaults torch.load(weights_only=True), which
# rejects legacy-format checkpoints such as face-parsing's resnet18-5c106cde.pth (2017). These are
# the official weights from pytorch.org / MuseTalk's HF repo, so full loading is acceptable here.
cmd = (f"cd {MT} && TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 python -m scripts.realtime_profile "
       "--inference_config configs/inference/realtime.yaml "
       "--result_dir results/realtime --unet_model_path models/musetalkV15/unet.pth "
       "--unet_config models/musetalkV15/musetalk.json --version v15 --fps 25 "
       "--batch_size 20 --skip_save_images")
r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
sampler.terminate()
print("exit", r.returncode)
print("\n".join(l for l in (r.stdout + r.stderr).splitlines()
                if any(k in l for k in ("PROFILE", "Total process", "Error", "Traceback")))[-3000:])
mem = [int(x) for x in open(mem_log).read().split() if x.isdigit()]
print(f"peak GPU memory: {max(mem) if mem else 'n/a'} MiB")
print("CPU cores:", subprocess.run("nproc", shell=True, capture_output=True, text=True).stdout.strip())

**Reading the result:** for each clip the script prints `Total process time of N frames … = T s`
(throughput = N / T fps; real time needs ≥ 25) and a `PROFILE` line with how long each concurrent
stage was busy: `gpu_busy` (Whisper features → UNet → VAE decode) and `blend_busy` (pasting each
generated face into the full frame, on one CPU thread). The stage whose busy time is close to T is
the bottleneck: GPU → split batches across both T4s; CPU → cheaper/parallel blending.

The first clip also includes one-time warm-up; the 1500-frame clip is the number that matters.

## Step 5c — our live renderer: one vs two GPUs, SD-VAE vs TAESD

MuseTalk's own script is GPU-bound at ~12 fps per T4 (`gpu_busy` ≈ total above). `trackb`'s
`MuseTalkRenderer` (the one the live agent uses) runs one model copy per GPU with batches
alternating between them, streams frames as each batch finishes, and can decode with TAESD.
This measures it on the avatar prepared above: throughput (`fps`, needs ≥ 25 — or ≥ 20 for a
20 fps avatar), time to first frame (adds to reply latency) and peak VRAM per GPU. The last cell
shows frame 50 from each decoder so quality can be compared by eye.

In [ ]:
REPO = "/kaggle/working/Rooman-Agent"
!rm -rf {REPO} && git clone -q --depth 1 -b feature/realtime-voice-loop https://github.com/daneshrooman-del/Rooman-Agent.git {REPO}
!cd {REPO} && git log --oneline -1
# trackb.avatar imports these (its package __init__ and trackb.tts -> trackb.config); checked in a
# clean venv: nothing heavier is pulled in.
!pip install -q livekit structlog pydantic-settings

In [ ]:
import os
env = dict(os.environ, PYTHONPATH=f"{REPO}/src", TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD="1")
configs = [("cuda:0", "sd-vae", 8), ("cuda:0,cuda:1", "sd-vae", 8), ("cuda:0,cuda:1", "taesd", 8)]
for devices, decoder, batch in configs:
    r = subprocess.run(["python", f"{REPO}/kaggle/musetalk_patches/bench_renderer.py", devices, decoder, str(batch)],
                       cwd=MT, env=env, capture_output=True, text=True)
    lines = [l for l in (r.stdout + r.stderr).splitlines() if l.startswith("RESULT")]
    print(lines[0] if lines else f"FAILED {devices} {decoder} (exit {r.returncode}):\n" + (r.stdout + r.stderr)[-2500:])

In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as mpimg
shots = [d for d in ("sd-vae", "taesd") if os.path.exists(f"/kaggle/working/frame_{d}.png")]
if not shots:
    print("no sample frames -- the renderer runs above failed; see their output")
else:
    fig, axes = plt.subplots(1, len(shots), figsize=(6 * len(shots), 6), squeeze=False)
    for ax, d in zip(axes[0], shots):
        ax.imshow(mpimg.imread(f"/kaggle/working/frame_{d}.png")); ax.set_title(d); ax.axis("off")
    plt.show()